In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os

# Updated path for Kaggle structure
data_dir = '/kaggle/input/et-autism-dataset' 

def verify_dataset_v2(src):
    categories = ['high', 'low', 'medium', 'mild']
    print(f"--- Dataset Report for: {src} ---")
    
    for cat in categories:
        # We add the extra 'cat' to the path to handle the double folder
        cat_path = os.path.join(src, cat, cat)
        
        if not os.path.exists(cat_path):
            print(f"❌ ERROR: Folder not found at {cat_path}")
            continue
            
        files = [f for f in os.listdir(cat_path) if f.lower().endswith('.jpg')]
        if len(files) == 0:
            print(f"⚠️ WARNING: No .jpg files found in {cat_path}")
        else:
            test_file = files[0]
            sub_id = test_file.split('-')[0]
            print(f"✅ Found {len(files)} images in {cat_path}. Sample ID: {sub_id}")

verify_dataset_v2(data_dir)

--- Dataset Report for: /kaggle/input/et-autism-dataset ---
✅ Found 1000 images in /kaggle/input/et-autism-dataset/high/high. Sample ID: 3
✅ Found 1000 images in /kaggle/input/et-autism-dataset/low/low. Sample ID: 14
✅ Found 1000 images in /kaggle/input/et-autism-dataset/medium/medium. Sample ID: 16
✅ Found 1000 images in /kaggle/input/et-autism-dataset/mild/mild. Sample ID: 44


In [2]:
import os
import shutil
import random
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score

# 1. SETUP
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

data_dir = '/kaggle/input/et-autism-dataset'
train_split_dir = '/kaggle/working/train_split'
test_split_dir = '/kaggle/working/test_split'

# 2. SUBJECT-WISE SPLIT (HANDLING DOUBLE FOLDERS)
def split_double_folders(src, train_dst, test_dst, split_ratio=0.8):
    categories = ['high', 'low', 'medium', 'mild']
    for cat in categories:
        # Pointing to the deep folder: /high/high/
        actual_source = os.path.join(src, cat, cat)
        if not os.path.exists(actual_source): continue
        
        files = [f for f in os.listdir(actual_source) if f.lower().endswith('.jpg')]
        subjects = list(set([f.split('-')[0] for f in files]))
        random.shuffle(subjects)
        
        limit = int(len(subjects) * split_ratio)
        train_subs = subjects[:limit]
        
        for f in files:
            sub_id = f.split('-')[0]
            dest_root = train_dst if sub_id in train_subs else test_dst
            target_folder = os.path.join(dest_root, cat)
            os.makedirs(target_folder, exist_ok=True)
            shutil.copy(os.path.join(actual_source, f), os.path.join(target_folder, f))
    print("✅ Split Complete. Data moved to /kaggle/working/")

# Run the split
split_double_folders(data_dir, train_split_dir, test_split_dir)

# 3. LOADERS
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255, brightness_range=[0.9, 1.1], horizontal_flip=False)

test_gen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_split_dir, target_size=(416, 416), batch_size=32, class_mode='categorical')

test_data = test_gen.flow_from_directory(
    test_split_dir, target_size=(416, 416), batch_size=32, class_mode='categorical', shuffle=False)

# 4. MODEL 1: CUSTOM CNN
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(416, 416, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    tf.keras.layers.GlobalAveragePooling2D(), # Prevents overfitting
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(4, activation='softmax')
])

# 5. OPTIMIZER & F1 CALLBACK
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(1e-3, 1000)
model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule),
              loss='categorical_crossentropy', metrics=['accuracy'])

class F1Metric(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        y_true = test_data.classes
        y_pred = np.argmax(self.model.predict(test_data, verbose=0), axis=1)
        score = f1_score(y_true, y_pred, average='macro')
        print(f" — val_f1_macro: {score:.4f}")

# 6. TRAIN
model.fit(train_data, validation_data=test_data, epochs=30, 
          callbacks=[F1Metric(), tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])

2026-01-14 09:45:12.017050: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768383912.243609      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768383912.306983      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768383912.832244      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768383912.832299      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768383912.832303      55 computation_placer.cc:177] computation placer alr

✅ Split Complete. Data moved to /kaggle/working/
Found 3200 images belonging to 4 classes.
Found 800 images belonging to 4 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1768383969.057366      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1768383969.061350      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can i

Epoch 1/30


I0000 00:00:1768383973.022150     139 service.cc:152] XLA service 0x7fa33420e880 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768383973.022192     139 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1768383973.022196     139 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1768383973.519855     139 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-01-14 09:46:19.182322: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-01-14 09:46:19.491179: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


  1/100 ━━━━━━━━━━━━━━━━━━━━ 20:52 13s/step - accuracy: 0.1875 - loss: 1.4015

I0000 00:00:1768383983.673091     139 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - accuracy: 0.2603 - loss: 1.3787 — val_f1_macro: 0.1747
100/100 ━━━━━━━━━━━━━━━━━━━━ 50s 377ms/step - accuracy: 0.2605 - loss: 1.3786 - val_accuracy: 0.2450 - val_loss: 1.3821
Epoch 2/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step - accuracy: 0.3762 - loss: 1.3243 — val_f1_macro: 0.3040
100/100 ━━━━━━━━━━━━━━━━━━━━ 35s 346ms/step - accuracy: 0.3761 - loss: 1.3240 - val_accuracy: 0.3600 - val_loss: 1.3556
Epoch 3/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - accuracy: 0.4089 - loss: 1.2188 — val_f1_macro: 0.3840
100/100 ━━━━━━━━━━━━━━━━━━━━ 35s 347ms/step - accuracy: 0.4091 - loss: 1.2186 - val_accuracy: 0.4087 - val_loss: 1.3388
Epoch 4/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 257ms/step - accuracy: 0.4743 - loss: 1.1623 — val_f1_macro: 0.3857
100/100 ━━━━━━━━━━━━━━━━━━━━ 35s 348ms/step - accuracy: 0.4742 - loss: 1.1623 - val_accuracy: 0.4062 - val_loss: 1.3475
Epoch 5/30
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 263ms/step - accuracy: 0.4709 - loss: 1.1318

KeyboardInterrupt: 

In [ ]:
import os
import shutil
import random
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score

# 1. SETUP REPRODUCIBILITY
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# PATHS (Ensure these are set for your Kaggle /working directory)
train_split_dir = '/kaggle/working/train_split'
test_split_dir = '/kaggle/working/test_split'

# 2. AGGRESSIVE DATA AUGMENTATION
# We added zoom and shifts to make it harder for the model to memorize pixels.
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    brightness_range=[0.8, 1.2],
    zoom_range=0.2,            # Randomly zoom in/out by 20%
    width_shift_range=0.1,     # Shift eye left/right
    height_shift_range=0.1,    # Shift eye up/down
    fill_mode='nearest',
    horizontal_flip=False      # Keep False to protect LE/RE labels
)

test_gen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_split_dir, target_size=(416, 416), batch_size=32, class_mode='categorical')

test_data = test_gen.flow_from_directory(
    test_split_dir, target_size=(416, 416), batch_size=32, class_mode='categorical', shuffle=False)

# 3. REWRITTEN ARCHITECTURE (Simpler + Batch Norm)
model = tf.keras.models.Sequential([
    # Block 1
    tf.keras.layers.Conv2D(32, (3, 3), padding='same', input_shape=(416, 416, 3)),
    tf.keras.layers.BatchNormalization(), # Stabilizes learning
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    
    # Block 2
    tf.keras.layers.Conv2D(64, (3, 3), padding='same'),
    tf.keras.layers.BatchNormalization(), 
    tf.keras.layers.Activation('relu'),
    tf.keras.layers.MaxPooling2D(2, 2),
    
    # Reduced Complexity: We removed a Conv layer to stop memorization
    tf.keras.layers.GlobalAveragePooling2D(), 
    
    # Final Classification
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.6), # High dropout to "kill" memorized features
    tf.keras.layers.Dense(4, activation='softmax')
])

# 4. OPTIMIZER WITH HIGHER LABEL SMOOTHING
# Smoothing 0.1 tells the model to be less "cocky" about its training data.
lr_schedule = tf.keras.optimizers.schedules.CosineDecay(1e-3, 2000)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-3)

model.compile(
    optimizer=optimizer, 
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), 
    metrics=['accuracy']
)

# 5. F1 MONITORING
class F1Metric(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        y_true = test_data.classes
        y_pred = np.argmax(self.model.predict(test_data, verbose=0), axis=1)
        score = f1_score(y_true, y_pred, average='macro')
        print(f" — val_f1_macro: {score:.4f}")

# 6. TRAIN
model.fit(
    train_data, 
    validation_data=test_data, 
    epochs=50, 
    callbacks=[F1Metric(), tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.metrics import f1_score

# 1. SETUP REPRODUCIBILITY
tf.keras.utils.set_random_seed(42) # Sets python, numpy, and tf seeds at once

# PATHS
train_split_dir = '/kaggle/working/train_split'
test_split_dir = '/kaggle/working/test_split'

# 2. PRE-TRAINED DATA AUGMENTATION
# Use the specific mobilenet_v2 preprocess_input function instead of rescale=1./255
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=applications.mobilenet_v2.preprocess_input,
    brightness_range=[0.8, 1.2],
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    fill_mode='nearest',
    horizontal_flip=False
)

test_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=applications.mobilenet_v2.preprocess_input
)

train_data = train_gen.flow_from_directory(
    train_split_dir, target_size=(416, 416), batch_size=32, class_mode='categorical')

test_data = test_gen.flow_from_directory(
    test_split_dir, target_size=(416, 416), batch_size=32, class_mode='categorical', shuffle=False)

# 3. BUILD TRANSFER LEARNING MODEL
base_model = applications.MobileNetV2(
    input_shape=(416, 416, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False # Freeze the "expert" brain

model2 = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5), # High dropout to prevent subject-wise memorization
    layers.Dense(4, activation='softmax')
])

# 4. COMPILE
# Using a slightly higher weight decay and label smoothing to fight overfitting
model2.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=0.0001, weight_decay=1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

# 5. F1 MONITORING CALLBACK
class F1Metric(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        y_true = test_data.classes
        y_pred = np.argmax(self.model.predict(test_data, verbose=0), axis=1)
        score = f1_score(y_true, y_pred, average='macro')
        print(f" — val_f1_macro: {score:.4f}")

# 6. TRAIN
model2.fit(
    train_data,
    validation_data=test_data,
    epochs=50,
    callbacks=[F1Metric(), tf.keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)]
)

model2.save("autism_severity_mobilenet.h5")

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications
from sklearn.metrics import f1_score

# 1. GPU STRATEGY SETUP
# Dual T4 GPU activation for faster parallel training.
strategy = tf.distribute.MirroredStrategy()
print(f'Number of devices: {strategy.num_replicas_in_sync}')

# 2. SETUP REPRODUCIBILITY
tf.keras.utils.set_random_seed(42)

# PATHS
train_split_dir = '/kaggle/working/train_split'
test_split_dir = '/kaggle/working/test_split'

# 3. AGGRESSIVE DATA AUGMENTATION
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=applications.mobilenet_v2.preprocess_input,
    brightness_range=[0.8, 1.2],
    zoom_range=0.3,
    width_shift_range=0.2,
    height_shift_range=0.2,
    fill_mode='nearest',
    horizontal_flip=False # Protected to maintain eye-type (le/re) context
)

test_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=applications.mobilenet_v2.preprocess_input
)

train_data = train_gen.flow_from_directory(
    train_split_dir, target_size=(416, 416), batch_size=32, class_mode='categorical')

test_data = test_gen.flow_from_directory(
    test_split_dir, target_size=(416, 416), batch_size=32, class_mode='categorical', shuffle=False)

# 4. BUILD MODEL WITHIN GPU SCOPE
with strategy.scope():
    # Base Model: MobileNetV2
    base_model = applications.MobileNetV2(
        input_shape=(416, 416, 3), include_top=False, weights='imagenet'
    )
    
    # --- UNFREEZING LOGIC ---
    base_model.trainable = True
    # Keep the early layers frozen, but allow the last 30 to adapt to gaze data
    for layer in base_model.layers[:-30]:
        layer.trainable = False
        
    # --- STACKED CLASSIFICATION HEAD ---
    model2 = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        
        # Block 1: Feature Extraction with L2 Regularization
        layers.Dense(512, kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5), # Suppresses participant-specific noise

        # Block 2: Refining Severity Markers
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.4),

        # Block 3: Final 4-Class Output
        layers.Dense(4, activation='softmax')
    ])

    # 5. COMPILE
    # Small learning rate prevents over-correction during fine-tuning.
    model2.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=1e-5, weight_decay=1e-3),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )

# 6. CALLBACKS: ADAPTIVE LEARNING & MONITORING
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1
)

class F1Metric(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        y_true = test_data.classes
        y_pred = np.argmax(self.model.predict(test_data, verbose=0), axis=1)
        score = f1_score(y_true, y_pred, average='macro')
        print(f" — val_f1_macro: {score:.4f}")

# 7. TRAIN
model2.fit(
    train_data,
    validation_data=test_data,
    epochs=50,
    callbacks=[
        F1Metric(), 
        reduce_lr, 
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
    ]
)

model2.save("autism_severity_final_v2.keras")

In [ ]:
import numpy as np
# Run this after your data loaders are defined
print("Training Counts:", np.unique(train_data.classes, return_counts=True))
print("Validation Counts:", np.unique(test_data.classes, return_counts=True))

In [ ]:
from sklearn.metrics import classification_report

# 1. Get the actual labels and model predictions
y_true = test_data.classes
y_pred = np.argmax(model2.predict(test_data, verbose=0), axis=1)

# 2. Print the detailed report
print(classification_report(y_true, y_pred, target_names=list(test_data.class_indices.keys())))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Generate Predictions
y_true = test_data.classes
y_pred = np.argmax(model2.predict(test_data, verbose=0), axis=1)

# 2. Build Matrix
cm = confusion_matrix(y_true, y_pred)
class_names = list(test_data.class_indices.keys())

# 3. Plot Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Autism Severity Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# 1. GPU STRATEGY SETUP
strategy = tf.distribute.MirroredStrategy()
tf.keras.utils.set_random_seed(42)

# GLOBAL BATCH SIZE
# With 2 GPUs, a batch of 64 means 32 images per GPU.
BATCH_SIZE = 64 

# 2. DATA GENERATORS
train_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=applications.mobilenet_v2.preprocess_input,
    brightness_range=[0.8, 1.2], zoom_range=0.3,
    width_shift_range=0.2, height_shift_range=0.2,
    fill_mode='nearest', horizontal_flip=False
)
test_gen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=applications.mobilenet_v2.preprocess_input
)

train_data = train_gen.flow_from_directory(
    '/kaggle/working/train_split', target_size=(416, 416), 
    batch_size=BATCH_SIZE, class_mode='categorical'
)
test_data = test_gen.flow_from_directory(
    '/kaggle/working/test_split', target_size=(416, 416), 
    batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False
)

# 3. CONSOLIDATED MODEL BUILDING
with strategy.scope():
    base_model = applications.MobileNetV2(input_shape=(416, 416, 3), include_top=False, weights='imagenet')
    base_model.trainable = False 
    
    model2 = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.4),
        layers.Dense(4, activation='softmax')
    ])

    custom_weights = {0: 1.5, 1: 2.0, 2: 1.0, 3: 1.5} 

    # PHASE 1 COMPILE
    model2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
                   loss='categorical_crossentropy', metrics=['accuracy'])

# 4. TRAINING PHASE 1
print("\n--- PHASE 1 START ---")
model2.fit(train_data, validation_data=test_data, epochs=10, class_weight=custom_weights)

# 5. PHASE 2 UNFREEZE & RE-COMPILE
with strategy.scope():
    print("\n--- PHASE 2: UNFREEZING ---")
    base_model.trainable = True
    for layer in base_model.layers[:-80]:
        layer.trainable = False
        
    model2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
                   loss='categorical_crossentropy', metrics=['accuracy'])

# 6. FINAL FIT
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True)

model2.fit(
    train_data, validation_data=test_data,
    epochs=100, 
    class_weight=custom_weights,
    callbacks=[reduce_lr, early_stop]
)

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# 1. GPU STRATEGY SETUP
# Using a clean strategy to manage the dual T4 GPUs
strategy = tf.distribute.MirroredStrategy()
tf.keras.utils.set_random_seed(42)

# GLOBAL BATCH SIZE: 32 per GPU = 64 total
BATCH_SIZE = 64 
IMG_SIZE = (416, 416)

# 2. DATA LOADING (Modern tf.data API for stability)
# This replaces ImageDataGenerator to solve the Shape Mismatch
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/train_split',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/test_split',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# 3. PREPROCESSING PIPELINE
# MobileNetV2 scaling must happen inside the dataset pipeline for multi-GPU
def preprocess(image, label):
    image = applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

# 4. BUILD MODEL WITHIN GPU SCOPE
with strategy.scope():
    base_model = applications.MobileNetV2(
        input_shape=(416, 416, 3), include_top=False, weights='imagenet'
    )
    base_model.trainable = False 
    
    model2 = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(4, activation='softmax')
    ])

    # Weights for autism classes: 0:high, 1:low, 2:medium, 3:mild
    custom_weights = {0: 1.2, 1: 1.8, 2: 1.0, 3: 1.8} 

    model2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy', 
        metrics=['accuracy']
    )

# 5. PHASE 1: STABILIZE HEAD
print("\n--- PHASE 1: STABILIZING HEAD ---")
model2.fit(train_ds, validation_data=val_ds, epochs=20, class_weight=custom_weights)

# 6. PHASE 2: SURGICAL UNFREEZE
with strategy.scope():
    print("\n--- PHASE 2: SURGICAL UNFREEZING ---")
    base_model.trainable = True
    for layer in base_model.layers[:-30]: # Unfreeze only top 30
        layer.trainable = False
        
    model2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Tiny LR
        loss='categorical_crossentropy', 
        metrics=['accuracy']
    )

# 7. FINAL FIT
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=15, restore_best_weights=True)

model2.fit(
    train_ds, validation_data=val_ds,
    epochs=100, 
    class_weight=custom_weights,
    callbacks=[reduce_lr, early_stop]
)

model2.save("autism_severity_final.keras")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# 1. GPU STRATEGY & REPRODUCIBILITY
strategy = tf.distribute.MirroredStrategy()
tf.keras.utils.set_random_seed(42)

# GLOBAL SETTINGS
BATCH_SIZE = 64 
IMG_SIZE = (416, 416)

# 2. DATA LOADING (tf.data API for Multi-GPU Stability)
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/train_split',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/test_split',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)

def preprocess(image, label):
    image = applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

# 3. BUILD MODEL WITHIN GPU SCOPE
with strategy.scope():
    base_model = applications.MobileNetV2(
        input_shape=(416, 416, 3), include_top=False, weights='imagenet'
    )
    base_model.trainable = False 
    
    model2 = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        # FIX: Increased L2 to 0.05 to penalize memorization
        layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(0.05)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        # FIX: Pushed Dropout to 0.7 to stop subject-wise memorization
        layers.Dropout(0.7), 
        layers.Dense(4, activation='softmax')
    ])

    custom_weights = {0: 1.2, 1: 1.8, 2: 1.0, 3: 1.8} 

    model2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy', metrics=['accuracy']
    )

# 4. PHASE 1: LONGER STABILIZATION (20 Epochs)
print("\n--- PHASE 1: STABILIZING HEAD ---")
model2.fit(train_ds, validation_data=val_ds, epochs=20, class_weight=custom_weights)

# 5. PHASE 2: SURGICAL UNFREEZE
with strategy.scope():
    print("\n--- PHASE 2: SURGICAL UNFREEZING ---")
    base_model.trainable = True
    for layer in base_model.layers[:-15]: 
        layer.trainable = False
        
    model2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='categorical_crossentropy', metrics=['accuracy']
    )

# 6. FINAL FIT WITH AGGRESSIVE EARLY STOPPING
# FIX: Reduced patience to 5 to stop training before overfitting "kills" the model
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', 
    patience=5, 
    restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)

model2.fit(
    train_ds, validation_data=val_ds,
    epochs=100, 
    class_weight=custom_weights,
    callbacks=[reduce_lr, early_stop]
)

model2.save("autism_severity_final_60plus.keras")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications

# 1. GPU STRATEGY & REPRODUCIBILITY
strategy = tf.distribute.MirroredStrategy()
tf.keras.utils.set_random_seed(42)

BATCH_SIZE = 64 
IMG_SIZE = (416, 416)

# 2. DATA LOADING (Modern tf.data API)
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/train_split',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/test_split',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)

def preprocess(image, label):
    image = applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

# 3. BUILD MODEL WITHIN GPU SCOPE
with strategy.scope():
    # A. Data Augmentation Layer: Simulates gaze variety
    data_augmentation = tf.keras.Sequential([
        layers.RandomRotation(0.1), # Small rotations for head tilt
        layers.RandomZoom(0.15),     # Simulates distance variety
        layers.RandomBrightness(0.15), # Simulates lighting changes
        layers.RandomTranslation(height_factor=0.1, width_factor=0.1)
    ])

    base_model = applications.MobileNetV2(
        input_shape=(416, 416, 3), include_top=False, weights='imagenet'
    )
    base_model.trainable = False 
    
    model2 = models.Sequential([
        layers.Input(shape=(416, 416, 3)),
        data_augmentation, # Augmentation happens on GPU
        base_model,
        layers.GlobalAveragePooling2D(),
        # Heavy L2 and Dropout to stop subject memorization
        layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(0.05)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.7), 
        layers.Dense(4, activation='softmax')
    ])

    # Weights to focus on the weak Mild/Low classes
    custom_weights = {0: 1.2, 1: 1.8, 2: 1.0, 3: 1.8} 

    model2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy', metrics=['accuracy']
    )

# 4. PHASE 1: DEEP STABILIZATION (40 Epochs)
print("\n--- PHASE 1: DEEP STABILIZATION ---")
model2.fit(train_ds, validation_data=val_ds, epochs=40, class_weight=custom_weights)

# 5. PHASE 2: PROGRESSIVE UNFREEZING (2 layers at a time)
total_layers_to_unfreeze = 30
step_size = 2

for i in range(step_size, total_layers_to_unfreeze + 1, step_size):
    print(f"\n--- PHASE 2: UNFREEZING LAST {i} LAYERS ---")
    with strategy.scope():
        base_model.trainable = True
        for layer in base_model.layers[:-i]:
            layer.trainable = False
        
        # Ultra-low LR to prevent Gradient Shock
        model2.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=1e-6),
            loss='categorical_crossentropy', metrics=['accuracy']
        )
    
    # Short bursts to let weights settle
    model2.fit(train_ds, validation_data=val_ds, epochs=3, class_weight=custom_weights)

# 6. FINAL REFINEMENT
print("\n--- FINAL REFINEMENT ---")
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=10, restore_best_weights=True
)

model2.fit(
    train_ds, validation_data=val_ds,
    epochs=50, class_weight=custom_weights, callbacks=[early_stop]
)

model2.save("autism_severity_progressive_aug_final.keras")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications

strategy = tf.distribute.MirroredStrategy()
tf.keras.utils.set_random_seed(42)

BATCH_SIZE = 64 
IMG_SIZE = (416, 416)

# DATA LOADING
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/train_split',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/test_split',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)

def preprocess(image, label):
    image = applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

with strategy.scope():
    # FIX: Simpler augmentation to keep the eye in focus
    data_augmentation = tf.keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomBrightness(0.1)
    ])

    base_model = applications.MobileNetV2(input_shape=(416, 416, 3), include_top=False, weights='imagenet')
    base_model.trainable = False 
    
    model2 = models.Sequential([
        layers.Input(shape=(416, 416, 3)),
        data_augmentation,
        base_model,
        layers.GlobalAveragePooling2D(),
        # FIX: Lower L2 and Dropout so the model can actually breathe
        layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5), 
        layers.Dense(4, activation='softmax')
    ])

    custom_weights = {0: 1.2, 1: 1.8, 2: 1.0, 3: 1.8} 

    # FIX: Higher LR to kickstart learning
    model2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss='categorical_crossentropy', metrics=['accuracy']
    )

# 4. PHASE 1: STABILIZATION (30 Epochs)
print("\n--- PHASE 1: STABILIZING HEAD ---")
model2.fit(train_ds, validation_data=val_ds, epochs=30, class_weight=custom_weights)

# 5. PHASE 2: PROGRESSIVE UNFREEZING
total_layers_to_unfreeze = 30
step_size = 2

for i in range(step_size, total_layers_to_unfreeze + 1, step_size):
    print(f"\n--- PHASE 2: UNFREEZING LAST {i} LAYERS ---")
    with strategy.scope():
        base_model.trainable = True
        for layer in base_model.layers[:-i]:
            layer.trainable = False
        
        # Use 5e-6. 1e-6 was too slow.
        model2.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
            loss='categorical_crossentropy', metrics=['accuracy']
        )
    
    model2.fit(train_ds, validation_data=val_ds, epochs=2, class_weight=custom_weights)

# 6. FINAL REFINEMENT
print("\n--- FINAL REFINEMENT ---")
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)

model2.fit(
    train_ds, validation_data=val_ds,
    epochs=50, class_weight=custom_weights, callbacks=[early_stop]
)

model2.save("autism_severity_progressive_final_v3.keras")

In [4]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, applications

strategy = tf.distribute.MirroredStrategy()
tf.keras.utils.set_random_seed(42)

BATCH_SIZE = 64 
IMG_SIZE = (416, 416)

# DATA LOADING (Clean tf.data API)
train_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/train_split',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/working/test_split',
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical'
)

def preprocess(image, label):
    # Using the standard MobileNetV2 scaler [-1, 1]
    image = applications.mobilenet_v2.preprocess_input(image)
    return image, label

train_ds = train_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(preprocess).prefetch(tf.data.AUTOTUNE)

with strategy.scope():
    # Simple, safe augmentation
    data_augmentation = tf.keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
    ])

    base_model = applications.MobileNetV2(input_shape=(416, 416, 3), include_top=False, weights='imagenet')
    base_model.trainable = False 
    
    model2 = models.Sequential([
        layers.Input(shape=(416, 416, 3)),
        data_augmentation,
        base_model,
        layers.GlobalAveragePooling2D(),
        # FIX: Remove L2 and lower Dropout to let it learn
        layers.Dense(512, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.01)), 
        layers.BatchNormalization(),
        layers.Dropout(0.5), 
        layers.Dense(256, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.01)), 
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(4, activation='softmax')
    ])

    custom_weights = {0: 1.0, 1: 1.5, 2: 1.0, 3: 1.5} 

    # FIX: Use a stronger learning rate to break the 25% trap
    model2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy', metrics=['accuracy']
    )

# 4. PHASE 1: STABILIZATION (20 Epochs)
print("\n--- PHASE 1: STABILIZING HEAD ---")
model2.fit(train_ds, validation_data=val_ds, epochs=20, class_weight=custom_weights)

# 5. PHASE 2: PROGRESSIVE UNFREEZING (If and only if Phase 1 hits >50%)
# Run this part only if Phase 1 is successful.

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Found 3200 files belonging to 4 classes.
Found 800 files belonging to 4 classes.


/tmp/ipykernel_55/2037860736.py:36: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = applications.MobileNetV2(input_shape=(416, 416, 3), include_top=False, weights='imagenet')


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

--- PHASE 1: STABILIZING HEAD ---
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then 

I0000 00:00:1768384219.231744     138 cuda_dnn.cc:529] Loaded cuDNN version 91002


50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step - accuracy: 0.4454 - loss: 11.6727INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
50/50 ━━━━━━━━━━━━━━━━━━━━ 36s 476ms/step - accuracy: 0.4469 - loss: 11.6461 - val_accuracy: 0.3125 - val_loss: 8.5165
Epoch 2/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 410ms/step - accuracy: 0.6130 - loss: 7.7395 - val_accuracy: 0.2550 - val_loss: 7.2195
Epoch 3/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 395ms/step - accuracy: 0.6918 - loss: 5.9999 - val_accuracy: 0.2925 - val_loss: 5.9005
Epoch 4/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 399ms/step - accuracy: 0.6847 - loss: 4.9463 - val_accuracy: 0.3638 - val_loss: 5.2240
Epoch 5/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 395ms/step - accuracy: 0.7217 - loss: 4.0935 - val_accuracy: 0.4062 - val_loss: 5.1008
Epoch 6/20
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 412ms/step - accuracy: 0.7586 - loss: 3.4110 - val_accuracy: 0.3575 - val_loss: 4.5291
Epoch 7/20
50/50 ━━━━

In [5]:
with strategy.scope():
    # Keep current augmentation
    data_augmentation = tf.keras.Sequential([
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
    ])

    base_model = applications.MobileNetV2(input_shape=(416, 416, 3), include_top=False, weights='imagenet')
    base_model.trainable = False 
    
    model2 = models.Sequential([
        layers.Input(shape=(416, 416, 3)),
        data_augmentation,
        base_model,
        layers.GlobalAveragePooling2D(),
        # STABILIZED SINGLE LAYER HEAD
        layers.Dense(256, kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5), # Higher dropout for 32 subjects
        layers.Dense(4, activation='softmax')
    ])

    custom_weights = {0: 1.0, 1: 1.5, 2: 1.0, 3: 1.5} 

    model2.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        # ADDED LABEL SMOOTHING TO STOP OVERCONFIDENCE
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), 
        metrics=['accuracy']
    )

# CRITICAL CALLBACKS FOR STABILITY
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, verbose=1)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=8, restore_best_weights=True)

model2.fit(
    train_ds, validation_data=val_ds, 
    epochs=40, # Allow more time to settle
    class_weight=custom_weights, 
    callbacks=[reduce_lr, early_stop]
)

/tmp/ipykernel_55/2271380945.py:8: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = applications.MobileNetV2(input_shape=(416, 416, 3), include_top=False, weights='imagenet')


Epoch 1/40
INFO:tensorflow:Collective all_reduce tensors: 6 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
50/50 ━━━━━━━━━━━━━━━━━━━━ 32s 509ms/step - accuracy: 0.5064 - loss: 5.3999 - val_accuracy: 0.2763 - val_loss: 4.1768 - learning_rate: 0.0010
Epoch 2/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 408ms/step - accuracy: 0.7068 - loss: 3.2236 - val_accuracy: 0.2612 - val_loss: 3.3232 - learning_rate: 0.0010
Epoch 3/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 403ms/step - accuracy: 0.7578 - loss: 2.3660 - val_accuracy: 0.2850 - val_loss: 2.6442 - learning_rate: 0.0010
Epoch 4/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 425ms/step - accuracy: 0.7731 - loss: 1.9002 - val_accuracy: 0.2125 - val_loss: 2.5915 - learning_rate: 0.0010
Epoch 5/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 413ms/step - accuracy: 0.7953 - loss: 1.6115 - val_accuracy: 0.3088 - val_loss: 2.1920 - learning_rate: 0.0010
Epoch 6/40
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 418ms/step - accuracy: 0.7996 - l

In [6]:
# 1. ROLLBACK TO BEST WEIGHTS
# We use the weights where val_loss was 1.44 to avoid the overfitting from Epoch 24.
# Note: Ensure you have your model object 'model2' still in memory.

# 2. DEFINE SURGICAL PARAMETERS
total_layers_to_unfreeze = 30
step_size = 2
custom_weights = {0: 1.0, 1: 1.5, 2: 1.0, 3: 1.5} # Keeping your severity weights

# 3. THE PROGRESSIVE LOOP
for i in range(step_size, total_layers_to_unfreeze + 1, step_size):
    print(f"\n--- PHASE 2: UNFREEZING LAST {i} LAYERS ---")
    
    with strategy.scope():
        base_model.trainable = True
        # Freeze all layers except the very last 'i' layers
        for layer in base_model.layers[:-i]:
            layer.trainable = False
        
        # INCREASE REGULARIZATION: Force the model to stop memorizing subjects
        # We re-compile with an ultra-low LR to prevent Gradient Shock
        model2.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
            loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
            metrics=['accuracy']
        )
    
    # Short settlement period: 3 epochs per 2-layer step
    model2.fit(
        train_ds, 
        validation_data=val_ds, 
        epochs=3, 
        class_weight=custom_weights,
        verbose=1
    )

# 4. FINAL GLOBAL SETTLEMENT
print("\n--- FINAL GLOBAL REFINEMENT ---")
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', 
    patience=10, 
    restore_best_weights=True
)

model2.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=50, 
    class_weight=custom_weights, 
    callbacks=[early_stop]
)

model2.save("autism_severity_progressive_final_best.keras")


--- PHASE 2: UNFREEZING LAST 2 LAYERS ---
Epoch 1/3
INFO:tensorflow:Collective all_reduce tensors: 8 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
50/50 ━━━━━━━━━━━━━━━━━━━━ 31s 489ms/step - accuracy: 0.7576 - loss: 1.2733 - val_accuracy: 0.3700 - val_loss: 2.3284
Epoch 2/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 416ms/step - accuracy: 0.7814 - loss: 1.2274 - val_accuracy: 0.2988 - val_loss: 2.4436
Epoch 3/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 20s 399ms/step - accuracy: 0.8042 - loss: 1.2031 - val_accuracy: 0.2325 - val_loss: 2.5549

--- PHASE 2: UNFREEZING LAST 4 LAYERS ---
Epoch 1/3
INFO:tensorflow:Collective all_reduce tensors: 8 all_reduces, num_devices = 2, group_size = 2, implementation = CommunicationImplementation.NCCL, num_packs = 1
50/50 ━━━━━━━━━━━━━━━━━━━━ 30s 462ms/step - accuracy: 0.8081 - loss: 1.1946 - val_accuracy: 0.2237 - val_loss: 2.6247
Epoch 2/3
50/50 ━━━━━━━━━━━━━━━━━━━━ 21s 423ms/step - accuracy: 0.7939 - loss: 1.2